In [1]:
import pandas as pd
import time
import numpy as np
import pickle
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.inspection import permutation_importance as pfi
from mlxtend.feature_selection import SequentialFeatureSelector as sfs
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE

In [2]:
def split_scale(indep_X,dep_Y):
    X_train,X_test,Y_train,Y_test=train_test_split(indep_X,dep_Y,test_size=0.25,random_state=0)
    sc=StandardScaler()
    X_train=sc.fit_transform(X_train)
    X_test=sc.transform(X_test)
    return X_train,X_test,Y_train,Y_test

In [3]:
def cm_predict(classifier,X_test):
    y_pred=classifier.predict(X_test)
    from sklearn.metrics import confusion_matrix
    cm=confusion_matrix(Y_test,y_pred)
    from sklearn.metrics import accuracy_score
    from sklearn.metrics import classification_report
    Accuracy=accuracy_score(Y_test,y_pred)
    report=classification_report(Y_test,y_pred)
    return classifier,Accuracy,report,X_test,Y_test,cm

In [4]:
def logistic(X_train,Y_train,X_test):
    from sklearn.linear_model import LogisticRegression
    classifier=LogisticRegression(random_state=0)
    classifier.fit(X_train,Y_train)
    classifier,Accuracy,report,X_test,Y_test,cm=cm_predict(classifier,X_test)
    return classifier,Accuracy,report,X_test,Y_test,cm

In [5]:
def svm_linear(X_train,Y_train,X_test):
    from sklearn.svm import SVC
    classifier=SVC(kernel='rbf',random_state=0)
    classifier.fit(X_train,Y_train)
    classifier,Accuracy,report,X_test,Y_test,cm=cm_predict(classifier,X_test)
    return classifier,Accuracy,report,X_test,Y_test,cm

In [6]:
def Naive(X_train,Y_train,X_test):
    from sklearn.naive_bayes import GaussianNB
    classifier=GaussianNB()
    classifier.fit(X_train,Y_train)
    classifier,Accuracy,report,X_test,Y_test,cm=cm_predict(classifier,X_test)
    return classifier,Accuracy,report,X_test,Y_test,cm

In [7]:
def knn(X_train,Y_train,X_test):
    from sklearn.neighbors import KNeighborsClassifier
    classifier=KNeighborsClassifier(n_neighbors=5,metric='minkowski',p=2)
    classifier.fit(X_train,Y_train)
    classifier,Accuracy,report,X_test,Y_test,cm=cm_predict(classifier,X_test)
    return classifier,Accuracy,report,X_test,Y_test,cm

In [8]:
def Decision(X_train,Y_train,X_test):
    from sklearn.tree import DecisionTreeClassifier
    classifier=DecisionTreeClassifier(criterion='entropy',random_state=0)
    classifier.fit(X_train,Y_train)
    classifier,Accuracy,report,X_test,Y_test,cm=cm_predict(classifier,X_test)
    return classifier,Accuracy,report,X_test,Y_test,cm

In [9]:
def random(X_train,Y_train,X_test):
    from sklearn.ensemble import RandomForestClassifier
    classifier=RandomForestClassifier(n_estimators=10,criterion='entropy',random_state=0)
    classifier.fit(X_train,Y_train)
    classifier,Accuracy,report,X_test,Y_test,cm=cm_predict(classifier,X_test)
    return classifier,Accuracy,report,X_test,Y_test,cm

In [10]:
def selectkbest(indep_X,dep_Y,n):
    test=SelectKBest(score_func=chi2 , k=n)
    fit1=test.fit(indep_X,dep_Y)
    select_features=fit1.transform(indep_X)
    
    selected_features_mask = fit1.get_support()
    selected_column_names = indep_X.columns[selected_features_mask].tolist()
    print(selected_column_names)
    return select_features

In [11]:
def forward(indep_X,dep_Y,n):
    forward_list=[]
    RF=RandomForestClassifier(n_estimators=10,criterion='entropy',random_state=0)
    DT=DecisionTreeClassifier(criterion='gini',max_features='sqrt',splitter='best',random_state=0)
    forwardModelList=[RF,DT]
    for i in forwardModelList:
        print(i)
        log_forward=sfs(i, k_features=n, forward=True, floating=False,  scoring='accuracy', cv=5, n_jobs=1)
        #log_forward=sfs(i,max_features=n)
        log_fit=log_forward.fit(indep_X,dep_Y)
        log_forward_feature=log_fit.transform(indep_X)
        forward_list.append(log_forward_feature)
        
        selected_feature_names = log_fit.k_feature_names_

        print("Selected Feature Names:")
        print(selected_feature_names)
    return forward_list

In [12]:
def backward(indep_X,dep_Y,n):
    back_list=[]
    RF=RandomForestClassifier(n_estimators=10,criterion='entropy',random_state=0)
    DT=DecisionTreeClassifier(criterion='gini',max_features='sqrt',splitter='best',random_state=0)
    backModelList=[RF,DT]
    for i in backModelList:
        print(i)
        log_back=sfs(i, k_features=n, forward=False, floating=False,  scoring='accuracy', cv=5, n_jobs=1)
        #log_forward=sfs(i,max_features=n)
        log_fit=log_back.fit(indep_X,dep_Y)
        log_back_feature=log_fit.transform(indep_X)
        back_list.append(log_back_feature)
        
        selected_feature_names = log_fit.k_feature_names_

        print("Selected Feature Names:")
        print(selected_feature_names)
    return back_list

In [13]:
def PFI(indep_X,dep_Y,n):
    PFI_List=[]
    RF=RandomForestClassifier(n_estimators=10,criterion='entropy',random_state=0)
    DT=DecisionTreeClassifier(criterion='gini',max_features='sqrt',splitter='best',random_state=0)
    RFO=RF.fit(indep_X,dep_Y)
    DTO=DT.fit(indep_X,dep_Y)
    PFIModelList=[DTO,RFO]
    for i in PFIModelList:    
        result = pfi(i, indep_X, dep_Y, n_repeats=30, scoring='accuracy', random_state=0, n_jobs=1)
        feature_names = indep_X.columns
        importances = result.importances
        sorted_idx = result.importances_mean.argsort()[::-1]
        names = [feature_names[j] for j in sorted_idx]
        num_features_to_select = n
        selected_feature_names = names[:num_features_to_select]
        print(selected_feature_names)
        PFI_List.append(selected_feature_names)
    return PFI_List

In [14]:
def construct_df(dataframe,alog,asvml,aknn,anb,adec,arf):
    for number,idex in enumerate(dataframe.index):
        dataframe['Logistic'][idex]=alog[number]
        dataframe['SVML'][idex]=asvml[number]
        dataframe['KNN'][idex]=aknn[number]
        dataframe['NaiveBayes'][idex]=anb[number]
        dataframe['DecisionTree'][idex]=adec[number]
        dataframe['RandomForest'][idex]=arf[number]
    return dataframe

In [15]:
def selectk_classification(alog,asvml,aknn,anb,adec,arf):
    dataframe=pd.DataFrame(index=['ChiSquare'],columns=['Logistic','SVML','KNN','NaiveBayes','DecisionTree','RandomForest'])
    Table=construct_df(dataframe,alog,asvml,aknn,anb,adec,arf)
    return Table

In [16]:
def forward_classification(alog,asvml,aknn,anb,adec,arf):
    forwarddataframe=pd.DataFrame(index=['RandomForest','DecisionTree'],columns=['Logistic','SVML','KNN','NaiveBayes','DecisionTree','RandomForest'])
    Table=construct_df(forwarddataframe,alog,asvml,aknn,anb,adec,arf)
    return Table

In [17]:
def back_classification(alog,asvml,aknn,anb,adec,arf):
    backdataframe=pd.DataFrame(index=['RandomForest','DecisionTree'],columns=['Logistic','SVML','KNN','NaiveBayes','DecisionTree','RandomForest'])
    Table=construct_df(backdataframe,alog,asvml,aknn,anb,adec,arf)
    return Table

In [18]:
def PFI_classification(alog,asvml,aknn,anb,adec,arf):
    PFIdataframe=pd.DataFrame(index=['RandomForest','DecisionTree'],columns=['Logistic','SVML','KNN','NaiveBayes','DecisionTree','RandomForest'])
    Table=construct_df(PFIdataframe,alog,asvml,aknn,anb,adec,arf)
    return Table

In [19]:
dataset=pd.read_csv('preprocessedEarthquake.csv',index_col=None)
df=dataset
df.drop(['cdi','mmi'],axis=1,inplace=True)    # Science CDI and MMI are measured after earthquake occurance, it is removed from the data to get significant features
order={'green':0,'yellow':1,'orange':2,'red':3}
df['Encoded_Level'] = df['alert'].map(order)
#df=pd.get_dummies(df,drop_first=True)
indep_X=df[['magnitude', 'sig', 'dmin', 'gap', 'depth']]
dep_Y=df[['Encoded_Level']]
number=5

In [20]:
kbest=selectkbest(indep_X,dep_Y,number)
alog=[]
asvml=[]
aknn=[]
anb=[]
adec=[]
arf=[]

X_train,X_test,Y_train,Y_test=split_scale(kbest,dep_Y)
        
classifier,Accuracy,report,X_test,Y_test,cm=logistic(X_train,Y_train,X_test)
alog.append(Accuracy)
    
classifier,Accuracy,report,X_test,Y_test,cm=svm_linear(X_train,Y_train,X_test)
asvml.append(Accuracy)
    
classifier,Accuracy,report,X_test,Y_test,cm=knn(X_train,Y_train,X_test)
aknn.append(Accuracy)
    
classifier,Accuracy,report,X_test,Y_test,cm=Naive(X_train,Y_train,X_test)
anb.append(Accuracy)
    
classifier,Accuracy,report,X_test,Y_test,cm=Decision(X_train,Y_train,X_test)
adec.append(Accuracy)
    
classifier,Accuracy,report,X_test,Y_test,cm=random(X_train,Y_train,X_test)
arf.append(Accuracy)

result=selectk_classification(alog,asvml,aknn,anb,adec,arf)
result  

['magnitude', 'sig', 'dmin', 'gap', 'depth']


C:\Anaconda\envs\aiml\lib\site-packages\sklearn\utils\validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Anaconda\envs\aiml\lib\site-packages\sklearn\utils\validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Anaconda\envs\aiml\lib\site-packages\sklearn\neighbors\_classification.py:198: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
C:\Anaconda\envs\aiml\lib\site-packages\sklearn\utils\validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example 

,Logistic,SVML,KNN,NaiveBayes,DecisionTree,RandomForest
ChiSquare,0.807692,0.807692,0.778846,0.759615,0.769231,0.788462


In [21]:
forward_list=forward(indep_X,dep_Y,number)
alog=[]
asvml=[]
aknn=[]
anb=[]
adec=[]
arf=[]
for i in forward_list:
        X_train,X_test,Y_train,Y_test=split_scale(i,dep_Y)
        
        classifier,Accuracy,report,X_test,Y_test,cm=logistic(X_train,Y_train,X_test)
        alog.append(Accuracy)
    
        classifier,Accuracy,report,X_test,Y_test,cm=svm_linear(X_train,Y_train,X_test)
        asvml.append(Accuracy)
    
        classifier,Accuracy,report,X_test,Y_test,cm=knn(X_train,Y_train,X_test)
        aknn.append(Accuracy)
    
        classifier,Accuracy,report,X_test,Y_test,cm=Naive(X_train,Y_train,X_test)
        anb.append(Accuracy)
    
        classifier,Accuracy,report,X_test,Y_test,cm=Decision(X_train,Y_train,X_test)
        adec.append(Accuracy)
    
        classifier,Accuracy,report,X_test,Y_test,cm=random(X_train,Y_train,X_test)
        arf.append(Accuracy)

result=forward_classification(alog,asvml,aknn,anb,adec,arf)
result

RandomForestClassifier(criterion='entropy', n_estimators=10, random_state=0)


C:\Anaconda\envs\aiml\lib\site-packages\sklearn\model_selection\_validation.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
C:\Anaconda\envs\aiml\lib\site-packages\sklearn\model_selection\_validation.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
C:\Anaconda\envs\aiml\lib\site-packages\sklearn\model_selection\_validation.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
C:\Anaconda\envs\aiml\lib\site-packages\sklearn\model_selection\_validation.py:680: DataConversionWarning: A column-vector y was passed whe

C:\Anaconda\envs\aiml\lib\site-packages\sklearn\model_selection\_validation.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
C:\Anaconda\envs\aiml\lib\site-packages\sklearn\model_selection\_validation.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
C:\Anaconda\envs\aiml\lib\site-packages\sklearn\model_selection\_validation.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
C:\Anaconda\envs\aiml\lib\site-packages\sklearn\model_selection\_validation.py:680: DataConversionWarning: A column-vector y was passed whe

C:\Anaconda\envs\aiml\lib\site-packages\sklearn\model_selection\_validation.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
C:\Anaconda\envs\aiml\lib\site-packages\sklearn\model_selection\_validation.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
C:\Anaconda\envs\aiml\lib\site-packages\sklearn\model_selection\_validation.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
C:\Anaconda\envs\aiml\lib\site-packages\sklearn\model_selection\_validation.py:680: DataConversionWarning: A column-vector y was passed whe

Selected Feature Names:
('magnitude', 'sig', 'dmin', 'gap', 'depth')
DecisionTreeClassifier(max_features='sqrt', random_state=0)
Selected Feature Names:
('magnitude', 'sig', 'dmin', 'gap', 'depth')


C:\Anaconda\envs\aiml\lib\site-packages\sklearn\utils\validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Anaconda\envs\aiml\lib\site-packages\sklearn\utils\validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Anaconda\envs\aiml\lib\site-packages\sklearn\neighbors\_classification.py:198: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
C:\Anaconda\envs\aiml\lib\site-packages\sklearn\utils\validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example 

,Logistic,SVML,KNN,NaiveBayes,DecisionTree,RandomForest
RandomForest,0.807692,0.807692,0.778846,0.759615,0.769231,0.788462
DecisionTree,0.807692,0.807692,0.778846,0.759615,0.769231,0.788462


In [22]:
back_list=backward(indep_X,dep_Y,number)
alog=[]
asvml=[]
aknn=[]
anb=[]
adec=[]
arf=[]
for i in forward_list:
        X_train,X_test,Y_train,Y_test=split_scale(i,dep_Y)
        
        classifier,Accuracy,report,X_test,Y_test,cm=logistic(X_train,Y_train,X_test)
        alog.append(Accuracy)
    
        classifier,Accuracy,report,X_test,Y_test,cm=svm_linear(X_train,Y_train,X_test)
        asvml.append(Accuracy)
    
        classifier,Accuracy,report,X_test,Y_test,cm=knn(X_train,Y_train,X_test)
        aknn.append(Accuracy)
    
        classifier,Accuracy,report,X_test,Y_test,cm=Naive(X_train,Y_train,X_test)
        anb.append(Accuracy)
    
        classifier,Accuracy,report,X_test,Y_test,cm=Decision(X_train,Y_train,X_test)
        adec.append(Accuracy)
    
        classifier,Accuracy,report,X_test,Y_test,cm=random(X_train,Y_train,X_test)
        arf.append(Accuracy)
        
result=back_classification(alog,asvml,aknn,anb,adec,arf)
result

C:\Anaconda\envs\aiml\lib\site-packages\sklearn\model_selection\_validation.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
C:\Anaconda\envs\aiml\lib\site-packages\sklearn\model_selection\_validation.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)


RandomForestClassifier(criterion='entropy', n_estimators=10, random_state=0)


C:\Anaconda\envs\aiml\lib\site-packages\sklearn\model_selection\_validation.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
C:\Anaconda\envs\aiml\lib\site-packages\sklearn\model_selection\_validation.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
C:\Anaconda\envs\aiml\lib\site-packages\sklearn\model_selection\_validation.py:680: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
C:\Anaconda\envs\aiml\lib\site-packages\sklearn\utils\validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d arra

Selected Feature Names:
('magnitude', 'sig', 'dmin', 'gap', 'depth')
DecisionTreeClassifier(max_features='sqrt', random_state=0)
Selected Feature Names:
('magnitude', 'sig', 'dmin', 'gap', 'depth')


C:\Anaconda\envs\aiml\lib\site-packages\sklearn\utils\validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Anaconda\envs\aiml\lib\site-packages\sklearn\neighbors\_classification.py:198: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
C:\Anaconda\envs\aiml\lib\site-packages\sklearn\utils\validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Anaconda\envs\aiml\lib\site-packages\ipykernel_launcher.py:4: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using rav

,Logistic,SVML,KNN,NaiveBayes,DecisionTree,RandomForest
RandomForest,0.807692,0.807692,0.778846,0.759615,0.769231,0.788462
DecisionTree,0.807692,0.807692,0.778846,0.759615,0.769231,0.788462


In [23]:
PFI_df=PFI(indep_X,dep_Y,number)
alog=[]
asvml=[]
asvmnl=[]
aknn=[]
anb=[]
adec=[]
arf=[]
PFI_0=PFI_df[0]
PFI_1=PFI_df[1]
Final=(PFI_0,PFI_1)
for i in Final:
    X=indep_X[i]
    X_train,X_test,Y_train,Y_test=split_scale(X,dep_Y)

    classifier,Accuracy,report,X_test,Y_test,cm=logistic(X_train,Y_train,X_test)
    alog.append(Accuracy)

    classifier,Accuracy,report,X_test,Y_test,cm=svm_linear(X_train,Y_train,X_test)
    asvml.append(Accuracy)

    classifier,Accuracy,report,X_test,Y_test,cm=knn(X_train,Y_train,X_test)
    aknn.append(Accuracy)

    classifier,Accuracy,report,X_test,Y_test,cm=Naive(X_train,Y_train,X_test)
    anb.append(Accuracy)

    classifier,Accuracy,report,X_test,Y_test,cm=Decision(X_train,Y_train,X_test)
    adec.append(Accuracy)

    classifier,Accuracy,report,X_test,Y_test,cm=random(X_train,Y_train,X_test)
    arf.append(Accuracy)

result=PFI_classification(alog,asvml,aknn,anb,adec,arf) 
result

C:\Anaconda\envs\aiml\lib\site-packages\ipykernel_launcher.py:5: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  """


['sig', 'dmin', 'depth', 'magnitude', 'gap']
['sig', 'depth', 'gap', 'dmin', 'magnitude']


C:\Anaconda\envs\aiml\lib\site-packages\sklearn\utils\validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Anaconda\envs\aiml\lib\site-packages\sklearn\utils\validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Anaconda\envs\aiml\lib\site-packages\sklearn\neighbors\_classification.py:198: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
C:\Anaconda\envs\aiml\lib\site-packages\sklearn\utils\validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example 

,Logistic,SVML,KNN,NaiveBayes,DecisionTree,RandomForest
RandomForest,0.807692,0.807692,0.778846,0.759615,0.798077,0.798077
DecisionTree,0.807692,0.807692,0.778846,0.759615,0.807692,0.807692


The feature selection is done on 'magnitude', 'sig', 'dmin', 'gap', 'depth'. Two features: ‘cdi’ and ‘mmi’ are removed for feature selection, as it is collected after the occurrence of earthquake. Out of the four feature selection method, to select 3 features, forward selection and backward elimination provides best results and the selected features are: ‘magnitude’, ‘sig’, and ‘depth’.